# Anton calibration with vLLM + LMCache

Measures, on a real GPU with the real serving stack, the three rates Anton's cost model and simulator use:

| Rate | How it is measured |
|---|---|
| **prefill seconds** for N tokens | vLLM, prefix caching off, `max_tokens=1`, median of 3 |
| **restore seconds** (cache tier → GPU) | vLLM + LMCache (CPU backend): store a prompt, clear vLLM's GPU prefix cache, time the same prompt again |
| **offload seconds** (GPU → cache tier) | time of a cold request with LMCache storing, minus the same request without LMCache |

It writes `calibration_vllm.json` in the same shape as `bench/calibration.json`. Copy it into the repo and point the simulator at it.

**Runs free** on a Colab or Kaggle T4 (Runtime → Change runtime type → GPU). Each engine runs in its own subprocess so GPU memory is fully released between the two configurations.

If the install fails on the T4, read the error before trying anything else: recent vLLM releases may drop support for older GPUs. An L4 (Colab paid, or the AWS g6 box from the GPU plan) avoids that.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Unpinned on purpose: Colab's CUDA/torch moves often, and the newest vLLM and
# LMCache are the most likely to match it. The versions are printed and
# recorded in the output file.
!pip install -q vllm lmcache 2>&1 | tail -3
import importlib.metadata as md
print({p: md.version(p) for p in ["vllm", "lmcache", "torch"]})

In [ ]:
%%writefile measure.py
"""One engine configuration per process. Prints a JSON line per context length."""
import json, os, statistics, sys, time

MODEL = os.environ.get("MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
LENGTHS = [int(x) for x in os.environ.get("LENGTHS", "1000,2000,4000,8000,16000").split(",")]
USE_LMCACHE = sys.argv[1] == "lmcache"

if USE_LMCACHE:
    # CHUNK (tokens per LMCache chunk) is worth sweeping: small chunks mean
    # many small copies, which may be what limits restore to ~0.65 GB/s.
    os.environ.update({"LMCACHE_CHUNK_SIZE": os.environ.get("CHUNK", "256"), "LMCACHE_LOCAL_CPU": "True",
                       # GB of host memory for the tier; 32k tokens of a 7B model is 1.8 GB per prompt.
                       "LMCACHE_MAX_LOCAL_CPU_SIZE": os.environ.get("TIER_GB", "20")})

from vllm import LLM, SamplingParams
kw = dict(model=MODEL, max_model_len=max(LENGTHS) + 256, gpu_memory_utilization=0.85,
          enable_prefix_caching=USE_LMCACHE, dtype="float16")
if USE_LMCACHE:
    from vllm.config import KVTransferConfig
    kw["kv_transfer_config"] = KVTransferConfig(kv_connector="LMCacheConnectorV1", kv_role="kv_both")
llm = LLM(**kw)
tok = llm.get_tokenizer()
one = SamplingParams(max_tokens=1, temperature=0)

def prompt(n, salt):
    # Distinct text per (length, salt) so nothing is cached by accident.
    words = (f"session {salt} line {i} of the agent transcript, tool output and code. " for i in range(n))
    ids = tok.encode(" ".join(words))[:n]
    return {"prompt_token_ids": ids}

def ttft(p):
    t = time.perf_counter()
    llm.generate([p], one, use_tqdm=False)
    return time.perf_counter() - t

ttft(prompt(256, "warmup"))
# KV bytes per token from the model's own config (stable across vLLM versions):
# K and V, per layer, per KV head, head_dim values, 2 bytes each in fp16.
from transformers import AutoConfig
hf = AutoConfig.from_pretrained(MODEL)
layers = hf.num_hidden_layers
kv_heads = getattr(hf, "num_key_value_heads", None) or hf.num_attention_heads
head_dim = getattr(hf, "head_dim", None) or hf.hidden_size // hf.num_attention_heads
bytes_per_token = 2 * layers * kv_heads * head_dim * 2

for n in LENGTHS:
    cold, restore = [], []
    for rep in range(3):
        p = prompt(n, f"{n}-{rep}")
        cold.append(ttft(p))              # lmcache: prefill + store into the CPU tier
        if USE_LMCACHE:
            # Drop vLLM's GPU copy (LMCache keeps the CPU copy). The reset is
            # refused while blocks are still in use, so retry until it reports
            # success; otherwise the "restore" would be a GPU cache hit.
            for _ in range(20):
                if llm.reset_prefix_cache():
                    break
                time.sleep(0.25)
            else:
                raise SystemExit(f"could not clear vLLM's GPU prefix cache at {n} tokens")
            restore.append(ttft(p))         # load from the tier instead of recomputing
    row = {"tokens": n, "cold_s": statistics.median(cold), "kv_bytes": n * bytes_per_token}
    if restore:
        row["restore_s"] = statistics.median(restore)
    print("ROW " + json.dumps(row), flush=True)

In [ ]:
import json, subprocess

def run(mode):
    out = subprocess.run(["python", "measure.py", mode], capture_output=True, text=True)
    rows = [json.loads(l[4:]) for l in out.stdout.splitlines() if l.startswith("ROW ")]
    if not rows:
        print(out.stderr[-3000:])
        raise SystemExit(f"{mode} run produced no measurements (see the error above)")
    return {r["tokens"]: r for r in rows}

plain = run("plain")     # vLLM alone: pure prefill
cached = run("lmcache")  # vLLM + LMCache: prefill+store, then restore
plain, cached

In [ ]:
import importlib.metadata as md, os, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
rows = []
for n in sorted(plain):
    prefill = plain[n]["cold_s"]
    rows.append({
        "tokens": n,
        "prefill_s": prefill,
        "kv_bytes": plain[n]["kv_bytes"],
        # Storing happens during the cold LMCache request; its extra time is the offload cost.
        "offload_s": max(0.0, cached[n]["cold_s"] - prefill),
        "restore_s": cached[n]["restore_s"],
    })
calib = {
    "model": os.environ.get("MODEL", "Qwen/Qwen2.5-1.5B-Instruct"),
    "stack": f"vLLM {md.version('vllm')} + LMCache {md.version('lmcache')} (CPU backend, chunk {os.environ.get('CHUNK', '256')}), fp16",
    "gpu": gpu,
    "rows": rows,
}
json.dump(calib, open("calibration_vllm.json", "w"), indent=1)
print(f"{'tokens':>7} {'prefill s':>10} {'restore s':>10} {'offload s':>10} {'restore/prefill':>16}")
for r in rows:
    print(f"{r['tokens']:>7} {r['prefill_s']:>10.3f} {r['restore_s']:>10.3f} {r['offload_s']:>10.3f} {r['restore_s']/r['prefill_s']:>16.2f}")

## What to look at

- **restore/prefill below 1** at a context length means moving the cache beats recomputing it there: the offload-vs-recompute rule should rescue those sessions.
- **The simulator's rates** come from the largest row: `PREFILL_MS_PER_TOKEN = prefill_s / tokens * 1000`, `BYTES_PER_TOKEN = kv_bytes / tokens`, `OFFLOAD_BYTES_PER_SEC = kv_bytes / offload_s`, `RESTORE_BYTES_PER_SEC = kv_bytes / restore_s`. Set them in `deploy/helm/anton/values.yaml` under `inference.sim`.
- LMCache's CPU backend lives on the same machine as the GPU. In the cluster the tier is across the network, so expect slower offload and restore there; the GPU-box run in the cloud plan measures that.

Download `calibration_vllm.json` (Files panel) and save it as `bench/calibration_vllm.json`.